In [1]:
import os
import pandas as pd
import numpy as np
import os.path as osp
from io import StringIO
folder = './raw'

qm = {}
annotations = pd.read_csv('annotations.tsv', sep='\t')
annotations.index = annotations['FILE']

for root, _, files in os.walk(folder):
    for file in files:
        if file == 'tblite-data.npz':
            quadrupole = np.load(osp.join(root, 'tblite-data.npz'))

            raw_xyz = []
            with open(osp.join(root, 'xtbopt.mol'), 'r') as f:
                while (line := f.readline()):
                    raw_xyz.append(line[:33])

            iron_index = list(pd.read_csv(StringIO('\n'.join(raw_xyz[4:int(raw_xyz[3][:3])+4])), sep='\\s+', names=['x','y','z','atom'])['atom'].values).index('Fe')
            
            #print(f"{root[11:]}.mol")
            qm[f"{root[6:]}.mol"] = [quadrupole[f'tblite0_qm_A_{x+13*' '}_alpha'][iron_index] -
                                     quadrupole[f'tblite0_qm_A_{x+13*' '}_beta'][iron_index]
                                     for x in ['xx','yy','zz','xy','xz','yz']]
            

annotations['qm'] = annotations['FILE'].map(qm)
#annotations.to_csv('./raw/annotations.tsv', index=False, sep='\t')

#annotations = pd.read_csv('asphericity_annotations.tsv', sep='\t')
qs_list = []
qs_sign = []
for EFG in annotations['qm'].values:
    qm_mat = np.asarray([[EFG[0],EFG[3],EFG[4]],
                         [EFG[3],EFG[1],EFG[5]],
                         [EFG[4],EFG[5],EFG[2]]])
    eigenvalues, _ = np.linalg.eig(qm_mat)
    sort = sorted(list(eigenvalues), key=abs, reverse=True)
    qs = -1.62026 * sort[0] * np.sqrt(1+(((sort[2]-sort[1])/sort[0])**2)/3)
    qs_list.append(qs)
    qs_sign.append(-1 if EFG[2] < 0 else 1)

from scipy.stats import pearsonr

pearsonr(annotations['QS'].values, qs_list).statistic

-0.23046490054960034

In [2]:
annotations['SIGN'] = qs_sign

In [3]:
annotations.drop('qm', axis=1).to_csv('sign_test.tsv', index=False, sep='\t')

In [8]:
import numpy as np
test = np.load('raw/185/tblite-data.npz')
list(test.keys())

['tblite0_bond-orders',
 'tblite0_CN_A',
 'tblite0_p_s_alpha',
 'tblite0_p_p_alpha',
 'tblite0_p_d_alpha',
 'tblite0_dipm_s_alpha',
 'tblite0_dipm_p_alpha',
 'tblite0_dipm_d_alpha',
 'tblite0_dipm_s_x            _alpha',
 'tblite0_dipm_s_y            _alpha',
 'tblite0_dipm_s_z            _alpha',
 'tblite0_dipm_p_x            _alpha',
 'tblite0_dipm_p_y            _alpha',
 'tblite0_dipm_p_z            _alpha',
 'tblite0_dipm_d_x            _alpha',
 'tblite0_dipm_d_y            _alpha',
 'tblite0_dipm_d_z            _alpha',
 'tblite0_qm_s_alpha',
 'tblite0_qm_p_alpha',
 'tblite0_qm_d_alpha',
 'tblite0_qm_s_xx             _alpha',
 'tblite0_qm_s_xy             _alpha',
 'tblite0_qm_s_yy             _alpha',
 'tblite0_qm_s_xz             _alpha',
 'tblite0_qm_s_yz             _alpha',
 'tblite0_qm_s_zz             _alpha',
 'tblite0_qm_p_xx             _alpha',
 'tblite0_qm_p_xy             _alpha',
 'tblite0_qm_p_yy             _alpha',
 'tblite0_qm_p_xz             _alpha',
 'tblite

In [9]:
sum(test['tblite0_p_s_alpha'] - test['tblite0_p_s_beta']) + sum(test['tblite0_p_p_alpha'] - test['tblite0_p_p_beta']) + sum(test['tblite0_p_d_alpha'] - test['tblite0_p_d_beta'])

4.000000002790823

In [85]:
[test[f'tblite0_qm_p_{x+13*' '}_alpha'][1] + 
                  test[f'tblite0_qm_p_{x+13*' '}_beta'][1] 
                  for x in ['zz','yy','xx','xy','xz','yz']]

[-0.05815187116189868,
 -0.016941213515566433,
 0.07509308467746356,
 0.15274787867716688,
 -0.021825147235769178,
 0.00820080233465256]

In [10]:
(test['tblite0_p_p_alpha'] + test['tblite0_p_p_beta'])[0]

1.584234322842517

In [11]:
(test['tblite0_p_d_alpha'] + test['tblite0_p_d_beta'])[0]

6.7320206325569965

In [12]:
(test['tblite0_p_s_alpha'] + test['tblite0_p_p_alpha'] + test['tblite0_p_d_alpha'] - (test['tblite0_p_s_beta'] + test['tblite0_p_p_beta'] + test['tblite0_p_d_beta']))[0]

3.1185725977839427

In [13]:
test['tblite0_p_s_alpha'] + test['tblite0_p_s_beta']

array([-0.191784  ,  1.46282923,  1.4628927 ,  1.46063388,  1.10173936,
        1.09883347,  1.04497549,  0.98360376,  0.97731742,  0.9887356 ,
        1.67280449,  1.45354748,  1.67717839,  1.04699999,  0.97822832,
        0.98009029,  0.98515193,  1.09839212,  1.04491694,  0.98248278,
        0.98299833,  0.98350898,  1.09935943,  1.10294813,  1.10267943,
        1.10093531,  1.04990297,  0.9493446 ,  0.95208273,  1.09976637,
        1.04656551,  0.97728574,  0.98228374,  0.98467064,  1.04300089,
        0.98606432,  0.98564554,  0.98199023,  1.04497806,  0.94602012,
        0.96165128,  1.10009096,  1.04674012,  0.98277541,  0.97313814,
        0.98176095,  1.09808597,  1.04349241,  0.98488454,  0.98251692,
        0.98513499,  1.04709815,  0.95885004,  0.9846757 ,  0.98181392,
        1.05531896,  0.97381458,  0.94352173,  0.98416695,  1.0557122 ,
        0.9520507 ,  0.97798574,  0.98804953,  1.0498539 ,  0.97730489,
        0.97620431,  0.95445627,  1.05029276,  0.95286601,  0.95